In [ ]:
# ============================================================
# ÉTAPE 1 : FUSION ET NETTOYAGE DES DONNÉES ATP (2020–2026)
# ============================================================

import pandas as pd
import numpy as np

# ─────────────────────────────────────────────
# 1. CHARGEMENT ET FUSION
# ─────────────────────────────────────────────

years = range(2020, 2027)
files = [f"../../data/tennis/atp_tennis_{year}.csv" for year in years]

# Chargement de chaque fichier avec ajout de la colonne 'year'
dfs = []
for year, file in zip(years, files):
    df_temp = pd.read_csv(file, header=0)  # force header=0 pour ignorer les doublons d'en-tête
    # Supprime les lignes où tourney_date vaut littéralement "tourney_date"
    df_temp = df_temp[df_temp["tourney_date"] != "tourney_date"]
    df_temp["year"] = year
    dfs.append(df_temp)

# Fusion en un seul DataFrame
df = pd.concat(dfs, ignore_index=True)

print(f"✅ Fusion terminée : {len(df):,} matchs | {df['year'].nunique()} saisons")
print(f"   Période : {df['year'].min()} → {df['year'].max()}")
df.head(3)

✅ Fusion terminée : 17,163 matchs | 7 saisons
   Période : 2020 → 2026


,tourney_id,tourney_name,surface,draw_size,tourney_level,indoor,tourney_date,match_num,winner_id,winner_seed,...,l_df,l_svpt,l_1stIn,l_1stWon,l_2ndWon,l_SvGms,l_bpSaved,l_bpFaced,year,tourney_id
0,2020-8888,ATP Cup,Hard,24,A,O,20200103,1,D632,NaN,...,2.0,116.0,72.0,42.0,18.0,17.0,6.0,14.0,2020,NaN
1,2020-8888,ATP Cup,Hard,24,A,O,20200103,2,GB88,NaN,...,2.0,63.0,39.0,23.0,10.0,9.0,7.0,11.0,2020,NaN
2,2020-8888,ATP Cup,Hard,24,A,O,20200103,3,N771,NaN,...,2.0,82.0,53.0,32.0,12.0,13.0,4.0,10.0,2020,NaN


In [3]:
# ─────────────────────────────────────────────
# 2. CONVERSION DES TYPES
# ─────────────────────────────────────────────

# tourney_date : entier YYYYMMDD → datetime
df["tourney_date"] = pd.to_datetime(df["tourney_date"].astype(str), format="%Y%m%d")

# Colonnes catégorielles
cat_cols = ["surface", "tourney_level", "round", "winner_hand", "loser_hand",
            "winner_ioc", "loser_ioc", "indoor"]
for col in cat_cols:
    df[col] = df[col].astype("category")

print("✅ Types convertis")
print(df.dtypes)

ValueError: time data "2020-01-03" doesn't match format "%Y%m%d". You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.

In [6]:
# ─────────────────────────────────────────────
# 3. ANALYSE DES VALEURS MANQUANTES
# ─────────────────────────────────────────────

missing = df.isnull().mean().mul(100).sort_values(ascending=False)
missing_signif = missing[missing > 0]

print("Colonnes avec valeurs manquantes (%) :\n")
print(missing_signif.to_string())

Colonnes avec valeurs manquantes (%) :

winner_entry          86.476723
loser_entry           79.817048
loser_seed            75.866690
winner_seed           58.830041
minutes                7.253977
indoor                 6.275127
w_SvGms                4.597098
l_SvGms                4.597098
l_bpSaved              4.538834
l_bpFaced              4.538834
l_2ndWon               4.533007
l_1stIn                4.533007
l_1stWon               4.533007
l_df                   4.533007
l_svpt                 4.533007
l_ace                  4.533007
w_2ndWon               4.533007
w_bpFaced              4.533007
w_bpSaved              4.533007
w_svpt                 4.527181
w_1stIn                4.527181
w_ace                  4.527181
w_df                   4.527181
w_1stWon               4.527181
match_num              2.849152
loser_rank_points      1.351745
loser_rank             1.351745
loser_ht               1.293480
loser_hand             0.879800
winner_rank            0.536037


In [8]:
# ─────────────────────────────────────────────
# 4. NETTOYAGE
# ─────────────────────────────────────────────

# --- 4a. Colonnes à fort taux de NaN (informations non pertinentes) ---
# winner_entry / loser_entry : ~80% NaN (qualification wildcard etc.)
# → on les conserve mais on remplit les NaN par "N" (joueur entré normalement)
df["winner_entry"] = df["winner_entry"].fillna("N")
df["loser_entry"]  = df["loser_entry"].fillna("N")

# winner_seed / loser_seed : NaN = joueur non tête de série → 0
# winner_seed / loser_seed : NaN = joueur non tête de série → 0
# pd.to_numeric avec errors='coerce' transforme les valeurs non-numériques ('Q', 'WC', 'LL'...) en NaN
df["winner_seed"] = pd.to_numeric(df["winner_seed"], errors="coerce").fillna(0).astype(int)
df["loser_seed"]  = pd.to_numeric(df["loser_seed"],  errors="coerce").fillna(0).astype(int)

# --- 4b. Colonnes de stats match (5% NaN) ---
# Ces NaN correspondent à des matchs retirés ou walkover (pas de stats)
# → On les identifie par un flag, puis on peut les filtrer ou imputer

stat_cols = [
    "minutes",
    "w_ace", "w_df", "w_svpt", "w_1stIn", "w_1stWon",
    "w_2ndWon", "w_SvGms", "w_bpSaved", "w_bpFaced",
    "l_ace", "l_df", "l_svpt", "l_1stIn", "l_1stWon",
    "l_2ndWon", "l_SvGms", "l_bpSaved", "l_bpFaced",
]

# Flag : 1 si le match a des stats complètes, 0 sinon
df["has_stats"] = df[stat_cols].notnull().all(axis=1).astype(int)

print(f"Matchs avec stats complètes  : {df['has_stats'].sum():,} ({df['has_stats'].mean():.1%})")
print(f"Matchs sans stats (W/O, RET) : {(df['has_stats'] == 0).sum():,}")

# --- 4c. Colonne 'indoor' ---
# ~8% NaN → on remplace par 'Unknown'
df["indoor"] = df["indoor"].cat.add_categories("Unknown").fillna("Unknown")

# --- 4d. Classement perdant manquant ---
# loser_rank NaN = joueur non classé → on impute avec un rang élevé fictif (999)
df["loser_rank"]        = df["loser_rank"].fillna(999).astype(int)
df["loser_rank_points"] = df["loser_rank_points"].fillna(0).astype(int)

# --- 4e. Suppression des doublons ---
before = len(df)
df = df.drop_duplicates()
print(f"\nDoublons supprimés : {before - len(df)}")

print("\n✅ Nettoyage terminé")
print(df.isnull().sum()[df.isnull().sum() > 0])  # vérification finale

Matchs avec stats complètes  : 15,861 (92.4%)
Matchs sans stats (W/O, RET) : 1,302

Doublons supprimés : 0

✅ Nettoyage terminé
surface                 53
draw_size               16
match_num              489
winner_hand             46
winner_ht               78
winner_ioc               2
winner_age               7
winner_rank             92
winner_rank_points      92
loser_id                 1
loser_hand             151
loser_ht               222
loser_ioc                4
loser_age                9
round                    1
minutes               1245
w_ace                  777
w_df                   777
w_svpt                 777
w_1stIn                777
w_1stWon               777
w_2ndWon               778
w_SvGms                789
w_bpSaved              778
w_bpFaced              778
l_ace                  778
l_df                   778
l_svpt                 778
l_1stIn                778
l_1stWon               778
l_2ndWon               778
l_SvGms                789
l_bpSave

In [9]:
# ─────────────────────────────────────────────
# 5. RÉSUMÉ FINAL
# ─────────────────────────────────────────────

print("=" * 45)
print(f"  Matchs totaux     : {len(df):,}")
print(f"  Colonnes          : {df.shape[1]}")
print(f"  Période           : {df['tourney_date'].min().date()} → {df['tourney_date'].max().date()}")
print(f"  Surfaces          : {df['surface'].cat.categories.tolist()}")
print(f"  Joueurs uniques   : {pd.concat([df['winner_name'], df['loser_name']]).nunique():,}")
print("=" * 45)

df.info()

  Matchs totaux     : 17,163
  Colonnes          : 52
  Période           : 2020-01-03 → 2026-04-19
  Surfaces          : ['Clay', 'Grass', 'Hard']
  Joueurs uniques   : 914
<class 'pandas.DataFrame'>
RangeIndex: 17163 entries, 0 to 17162
Data columns (total 52 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   tourney_id          17163 non-null  str           
 1   tourney_name        17163 non-null  str           
 2   surface             17110 non-null  category      
 3   draw_size           17147 non-null  object        
 4   tourney_level       17163 non-null  category      
 5   indoor              17163 non-null  category      
 6   tourney_date        17163 non-null  datetime64[us]
 7   match_num           16674 non-null  object        
 8   winner_id           17163 non-null  str           
 9   winner_seed         17163 non-null  int64         
 10  winner_entry        17163 non-null  str           


In [ ]:
# ─────────────────────────────────────────────
# 6. SAUVEGARDE
# ─────────────────────────────────────────────

df.to_csv("../../data/tennis/atp_clean.csv", index=False)
print("✅ Fichier sauvegardé : atp_clean.csv")

✅ Fichier sauvegardé : atp_clean.csv
